# PyroClass 3-Class XGBoost Training V2 - Final

**Status**: Optimized baseline (no hyperparameter tuning)

**Why no tuning?**
- Weak labels have temporal bias
- Tuning weak labels = optimizing noise
- 2 hours tuning → 1% improvement (not worth it)
- Better to fix label generation

**Time**: ~5 minutes

**Setup**:
1. Upload 3 V2 preprocessed CSV files to Colab
2. Run all cells in order
3. Download 11 artifacts

**Classes**: forest_fire, non_industrial, unknown (3-class prototype)

## Step 1: Upload Preprocessed V2 Data

In [ ]:
from google.colab import files

print("Select the 3 V2 preprocessed CSV files:")
print("  - pyroclass_train_preprocessed_v2.csv")
print("  - pyroclass_validation_preprocessed_v2.csv")
print("  - pyroclass_test_preprocessed_v2.csv")
print()
uploaded = files.upload()

## Step 2: Train Model (No Tuning, ~5 minutes)

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_recall_fscore_support
import shap
import pickle
import json
import time
import os
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("PYROCLASS 3-CLASS XGBOOST TRAINING - V2 FINAL (NO TUNING)")
print("="*80)

# Load data
print("\n[PHASE 1] Loading optimized preprocessed data...")
train_df = pd.read_csv('pyroclass_train_preprocessed_v2.csv')
val_df = pd.read_csv('pyroclass_validation_preprocessed_v2.csv')
test_df = pd.read_csv('pyroclass_test_preprocessed_v2.csv')

print(f"✓ Train: {train_df.shape}")
print(f"✓ Val: {val_df.shape}")
print(f"✓ Test: {test_df.shape}")

# Prepare features
print("\n[PHASE 2] Preparing features...")
drop_cols = ['hotspot_id', 'latitude', 'longitude', 'timestamp', 'target_class', 
             'label_source', 'label_confidence', 'type', 'daynight', 'h3_cell', 'sample_weight']
feature_cols = [col for col in train_df.columns if col not in drop_cols]
print(f"✓ Features: {len(feature_cols)}")

X_train = train_df[feature_cols]
y_train = train_df['target_class']
sample_weight_train = train_df['sample_weight'].values
X_val = val_df[feature_cols]
y_val = val_df['target_class']
X_test = test_df[feature_cols]
y_test = test_df['target_class']

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_val_encoded = le.transform(y_val)
y_test_encoded = le.transform(y_test)

print(f"✓ Labels: {dict(zip(le.classes_, range(len(le.classes_))))}")

# Train model
print("\n[PHASE 3] Training XGBoost (optimized defaults, no tuning)...")
start_time = time.time()

model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    objective='multi:softprob',
    num_class=3,
    tree_method='hist',
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss',
    early_stopping_rounds=50,
    reg_alpha=0.1,
    reg_lambda=1.0,
    subsample=0.8,
    colsample_bytree=0.8,
    verbosity=0
)

model.fit(
    X_train, y_train_encoded,
    sample_weight=sample_weight_train,
    eval_set=[(X_train, y_train_encoded), (X_val, y_val_encoded)],
    verbose=100
)

training_time = time.time() - start_time
print(f"\n✓ Training complete in {training_time/60:.2f} minutes")
print(f"✓ Best iteration: {model.best_iteration}")

# Evaluate
print("\n[PHASE 4] Evaluating...")
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test_encoded, y_pred)
macro_f1 = f1_score(y_test_encoded, y_pred, average='macro')
precision, recall, f1, support = precision_recall_fscore_support(y_test_encoded, y_pred)

print(f"✓ Test Accuracy: {accuracy:.4f}")
print(f"✓ Macro F1: {macro_f1:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test_encoded, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_test_encoded, y_pred)
print(f"\nConfusion Matrix:")
print(cm)

# Feature importance
print("\n[PHASE 5] Feature importance...")
feature_importance = model.feature_importances_
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

print(f"\nTop 15 Features:")
print(importance_df.head(15).to_string(index=False))

# Visualizations
print("\n[PHASE 6] Creating visualizations...")
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ Saved: confusion_matrix.png")

plt.figure(figsize=(10, 8))
importance_df.head(20).plot(x='feature', y='importance', kind='barh', figsize=(10, 8))
plt.title('Top 20 Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ Saved: feature_importance.png")

# SHAP
print("\n[PHASE 7] Computing SHAP explanations (2-3 minutes)...")
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test, feature_names=feature_cols, class_names=le.classes_, show=False)
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ Saved: shap_summary.png")

print("\n" + "="*80)
print("✅ TRAINING COMPLETE")
print("="*80)

## Step 3: Export Artifacts & Download

In [ ]:
print("[PHASE 8] Exporting artifacts...\n")

# Save model
model.save_model('xgboost_model.json')
with open('xgboost_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print("✓ Saved: xgboost_model.pkl")

# Save encoder
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
print("✓ Saved: label_encoder.pkl")

# Save explainer
with open('shap_explainer.pkl', 'wb') as f:
    pickle.dump(explainer, f)
print("✓ Saved: shap_explainer.pkl")

# Save schema
feature_schema = {
    "version": "v2-final",
    "num_features": len(feature_cols),
    "feature_names": feature_cols,
    "encoding": {"label_mapping": {name: int(idx) for idx, name in enumerate(le.classes_)}}
}
with open('feature_schema.json', 'w') as f:
    json.dump(feature_schema, f, indent=2)
print("✓ Saved: feature_schema.json")

# Save metadata
per_class_metrics = {}
precision, recall, f1, support = precision_recall_fscore_support(y_test_encoded, y_pred)
for i, class_name in enumerate(le.classes_):
    per_class_metrics[class_name] = {
        "precision": float(precision[i]),
        "recall": float(recall[i]),
        "f1": float(f1[i])
    }

metadata = {
    "version": "v2.0.0-final",
    "training_time_minutes": round(training_time/60, 2),
    "accuracy": float(accuracy),
    "macro_f1": float(macro_f1),
    "per_class_metrics": per_class_metrics,
    "hyperparameter_tuning": "SKIPPED",
    "rationale": "Weak labels with temporal bias. Tuning would optimize noise, not real patterns."
}

with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print("✓ Saved: model_metadata.json")

# Save importance
importance_df.to_csv('feature_importance.csv', index=False)
print("✓ Saved: feature_importance.csv")

# Download
print("\n[PHASE 9] Downloading artifacts...\n")
artifacts = [
    'xgboost_model.pkl', 'xgboost_model.json', 'label_encoder.pkl', 'shap_explainer.pkl',
    'feature_schema.json', 'model_metadata.json', 'feature_importance.csv',
    'confusion_matrix.png', 'feature_importance.png', 'shap_summary.png'
]

for artifact in artifacts:
    if os.path.exists(artifact):
        files.download(artifact)
        print(f"✓ {artifact}")

print("\n" + "="*80)
print("✅ ALL ARTIFACTS DOWNLOADED")
print("="*80)
print(f"\nSummary:")
print(f"  Training Time: {training_time/60:.2f} minutes")
print(f"  Test Accuracy: {accuracy:.4f}")
print(f"  Macro F1: {macro_f1:.4f}")
print(f"  Artifacts: {len(artifacts)} files")
print(f"\nNext: Send to backend engineer for integration")